# Colab quickstart and run manager for the TRPO / PPO repo
This notebook is a **reference / helper notebook** for running this repository on Google Colab.
It is designed around the following workflow I settled upon while working on the development of this project:
1. keep the canonical repo on Google Drive
2. copy it into `/content/trpo` at the start of a Colab session
3. run experiments from local SSD (`/content`)
4. optionally save outputs back to Drive afterward


This notebook starts with a Colab quickstart flow, after which we have added a **Run Manager layer** so we can launch training runs without retyping long commands.
The usual flow is:
1. Run sections 1-7 once to mount Drive, copy the repo, install dependencies, configure paths, define helpers, and load argument presets.
2. Use **section 8** for the important one-off training call: one environment, one policy/config, one seed.
3. Use **section 9** when you want to batch over several configs or seeds.
4. Use **section 10** after training for aggregation and checkpoint evaluation.

The helper cell is intentionally standalone, so the notebook still works by itself in Colab. You normally do not edit it; edit the launcher cells instead.

The examples below are intentionally simple and meant to be copied / adapted.

## 1. Mount Google Drive

In [ ]:

from google.colab import drive

drive.mount('/content/drive')


## 2. Copy the repo from Drive to Colab local SSD

Edit the Drive path below to match your own folder structure.

In [ ]:

# Adjust these paths if your Drive layout is different.
DRIVE_REPO_PATH = "/content/drive/MyDrive/Colab Notebooks/839/trpo"
LOCAL_REPO_PATH = "/content/trpo"

!rm -rf "$LOCAL_REPO_PATH"
!cp -r "$DRIVE_REPO_PATH" "$LOCAL_REPO_PATH"
%cd "$LOCAL_REPO_PATH"


## 3. Install dependencies inside the Colab runtime

The repository scripts bootstrap the local `/content/trpo` checkout, so the dependency install is enough when running commands from the repo root.

In [ ]:

!python3 -m pip install -r requirements.txt


## 4. Check CUDA and available CPU resources

In [ ]:

import os
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('CPU count:', os.cpu_count())


### 4.1/4 Example: locomotion run

This is a simple example for a locomotion task.

In [ ]:
# !python3 scripts/train.py \
# --config configs/mujoco/swimmer_single_path.yaml \
# --device cuda \
# --progress-mode notebook --overwrite

### 4.2/4 Example Atari TRPO run on Colab

This uses the practical Colab setup we saw that worked well during development:

- `memory_mode safe`
- `obs_storage ram`
- tuned `full_batch_chunk_size`
- optional Fisher-vector-product subsampling
- CUDA
- local output directory under `/content/...`

In [ ]:
# !python3 scripts/train.py \
#    --config configs/atari/seaquest_single_path.yaml \
#    --memory-mode safe   --obs_storage ram \
#    --full_batch_chunk_size 8192   --fvp_subsample_fraction 0.1 \
#    --device cuda   --progress-mode off \
#    --output-dir /content/trpo_runs/seaquest_single_path/seed_0   --overwrite

### 4.3/4 Example: Atari PPO run on Colab

In [ ]:
# !python3 scripts/train.py \
#    --config configs/atari/seaquest_ppo_clip. \
#    --memory-mode standard   --device cuda   --progress-mode off \
#    --output-dir /content/trpo_runs/seaquest_ppo_clip/seed_0   --overwrite

### 4.7/8 Batch launcher reproducing the 2015 TRPO paper implementations

This preserves the lightweight paper-batch launcher that used to live in a standalone script.

In [ ]:
# import subprocess
# import sys

# PAPER_SUITES = {
#     "mujoco": [
#         "configs/mujoco/swimmer_single_path.yaml",
#         "configs/mujoco/hopper_single_path.yaml",
#         "configs/mujoco/walker2d_single_path.yaml",
#     ],
#     "atari": [
#         "configs/atari/beamrider_single_path.yaml",
#         "configs/atari/breakout_single_path.yaml",
#         "configs/atari/enduro_single_path.yaml",
#         "configs/atari/pong_single_path.yaml",
#         "configs/atari/qbert_single_path.yaml",
#         "configs/atari/seaquest_single_path.yaml",
#         "configs/atari/spaceinvaders_single_path.yaml",
#     ],
# }

# def run_paper_suite(suite, seeds=(0,), device="cuda", extra_args=()):
#     for config in PAPER_SUITES[suite]:
#         for seed in seeds:
#             cmd = [
#                 sys.executable,
#                 "scripts/train.py",
#                 "--config",
#                 config,
#                 "--seed",
#                 str(seed),
#                 "--device",
#                 device,
#                 *extra_args,
#             ]
#             print("Running:", " ".join(cmd))
#             subprocess.run(cmd, check=True)

# # Uncomment one when you are ready.
# # run_paper_suite("mujoco", seeds=(0,), device="cuda")
# # run_paper_suite("atari", seeds=(0,), device="cuda", extra_args=("--memory-mode", "safe", "--obs-storage", "ram", "--progress-mode", "off"))

### 4.15/16 Aggregate runs

In [ ]:
# !python3 scripts/aggregate_results.py \
#    --runs-root outputs/swimmer_single_path \
#    --runs-root outputs/swimmer_natural_pg \
#    --runs-root outputs/swimmer_ppo_clip \
#    --compare   --metric train_return_mean   --x-axis iteration

### 4.15999/16 Copying results back to drive

If you wrote outputs to `/content/...`, copy them back to Drive when the run finishes.

In [ ]:
# !mkdir -p "/content/drive/MyDrive/Colab Notebooks/839/trpo_outputs"
# !cp -r /content/trpo_runs "/content/drive/MyDrive/Colab Notebooks/839/trpo_outputs/"


## 5. Configure default paths

These are the two output roots you will care about most:

- `LOCAL_RUNS_ROOT`: fast Colab SSD location used during training
- `DRIVE_RUNS_ROOT`: Google Drive location where finished and in-progress runs get synced


In [ ]:

from pathlib import Path

LOCAL_RUNS_ROOT = Path('/content/trpo_runs')
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/839/trpo_outputs')

LOCAL_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print('Local runs root :', LOCAL_RUNS_ROOT)
print('Drive runs root :', DRIVE_RUNS_ROOT)


## 6. Run Manager helpers

Run this cell once after the setup cells above. It defines the notebook API:

- `run_training(...)` launches one training run and syncs its output folder to Drive.
- `run_training(..., resume_from=...)` resumes from a specific checkpoint while keeping the same sync behavior.
- `resume_training(...)` is the convenience wrapper for checkpoint resumes.
- `checkpoint_path_for_run(...)` finds `epoch_XXXX.pt`, `last.pt`, or the latest checkpoint in a local/Drive run folder.
- `stitch_resumed_run(...)` compiles an original partial run plus a resumed run into one graphable seed folder.
- `run_batch(...)` loops over a batch plan and calls `run_training(...)` for each config/seed pair.
- `aggregate_runs(...)` and `evaluate_checkpoint(...)` are post-run utilities.

Everything starting with `_` is internal glue for command construction, syncing, and checkpoint aliases. The launch cells below are the ones you should usually edit.


In [ ]:
import csv
import json
import os
import shutil
import signal
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

REPO_ROOT = Path('/content/trpo')
TRAIN_SCRIPT = REPO_ROOT / 'scripts' / 'train.py'
EVAL_SCRIPT = REPO_ROOT / 'scripts' / 'evaluate.py'
AGGREGATE_SCRIPT = REPO_ROOT / 'scripts' / 'aggregate_results.py'
STITCH_SCRIPT = REPO_ROOT / 'scripts' / 'stitch_resumed_run.py'


def _cli_value(value: Any) -> str:
    if isinstance(value, bool):
        return 'true' if value else 'false'
    return str(value)


def _add_cli_args(cmd: list[str], extra_args: dict[str, Any] | None) -> list[str]:
    if not extra_args:
        return cmd
    for key, value in extra_args.items():
        key = str(key)
        if not key.startswith('--'):
            raise ValueError(f'Expected CLI key starting with --, got: {key}')
        if value is None:
            continue
        if isinstance(value, bool):
            if value:
                cmd.append(key)
        else:
            cmd.extend([key, _cli_value(value)])
    return cmd


def _show_command(cmd: list[str]) -> None:
    separator = ' ' + '\\' + '\n  '
    print(separator.join(cmd))


def _unique_paths(paths: Iterable[Path]) -> list[Path]:
    seen = set()
    unique = []
    for path in paths:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        unique.append(path)
    return unique


def _candidate_existing_paths(
    path: str | Path,
    local_runs_root: Path | str = LOCAL_RUNS_ROOT,
    drive_runs_root: Path | str = DRIVE_RUNS_ROOT,
) -> list[Path]:
    path = Path(path).expanduser()
    if path.is_absolute():
        return [path]
    return _unique_paths(
        [
            path,
            Path(local_runs_root) / path,
            Path(drive_runs_root) / path,
            REPO_ROOT / path,
        ]
    )


def _resolve_existing_path(
    path: str | Path,
    local_runs_root: Path | str = LOCAL_RUNS_ROOT,
    drive_runs_root: Path | str = DRIVE_RUNS_ROOT,
) -> Path:
    candidates = _candidate_existing_paths(path, local_runs_root, drive_runs_root)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def _checkpoint_epoch_hint(path: str | Path | None) -> int | None:
    if path is None:
        return None
    stem = Path(path).stem
    if not stem.startswith('epoch_'):
        return None
    try:
        return int(stem.removeprefix('epoch_'))
    except ValueError:
        return None


def _checkpoint_seed_dir(checkpoint_path: str | Path) -> Path | None:
    checkpoint_path = Path(checkpoint_path)
    if checkpoint_path.parent.name != 'checkpoints':
        return None
    return checkpoint_path.parent.parent


def _infer_seed_from_checkpoint(checkpoint_path: str | Path) -> int | None:
    seed_dir = _checkpoint_seed_dir(checkpoint_path)
    if seed_dir is None or not seed_dir.name.startswith('seed_'):
        return None
    raw_seed = seed_dir.name.removeprefix('seed_').split('_', 1)[0]
    try:
        return int(raw_seed)
    except ValueError:
        return None


def _infer_run_stem_from_checkpoint(checkpoint_path: str | Path) -> str | None:
    seed_dir = _checkpoint_seed_dir(checkpoint_path)
    if seed_dir is None:
        return None
    return seed_dir.parent.name or None


def _default_resume_output_tag(checkpoint_path: str | Path) -> str:
    epoch = _checkpoint_epoch_hint(checkpoint_path)
    return 'resume' if epoch is None else f'resume_{epoch:04d}'


def checkpoint_path_for_run(
    run_dir: str | Path,
    epoch: int | None = None,
    alias: str = 'last',
    local_runs_root: Path | str = LOCAL_RUNS_ROOT,
    drive_runs_root: Path | str = DRIVE_RUNS_ROOT,
) -> Path:
    """Return a checkpoint path from a local or Drive run folder.

    `run_dir` can be absolute or relative to either runs root, for example
    `pong_single_path/seed_0` or `DRIVE_RUNS_ROOT / 'pong_single_path' / 'seed_0'`.
    """
    run_dir = _resolve_existing_path(run_dir, local_runs_root, drive_runs_root)
    ckpt_dir = run_dir / 'checkpoints'
    if epoch is not None:
        checkpoint = ckpt_dir / f'epoch_{int(epoch):04d}.pt'
    else:
        checkpoint = ckpt_dir / f'{alias}.pt'
        if not checkpoint.exists():
            ckpts = sorted(ckpt_dir.glob('epoch_*.pt')) if ckpt_dir.exists() else []
            if ckpts:
                checkpoint = ckpts[-1]
    if not checkpoint.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint}')
    return checkpoint


def config_path_for_checkpoint(
    checkpoint_path: str | Path,
    local_runs_root: Path | str = LOCAL_RUNS_ROOT,
    drive_runs_root: Path | str = DRIVE_RUNS_ROOT,
) -> Path:
    checkpoint_path = _resolve_existing_path(checkpoint_path, local_runs_root, drive_runs_root)
    seed_dir = _checkpoint_seed_dir(checkpoint_path)
    if seed_dir is None:
        raise ValueError(f'Could not infer run directory from checkpoint path: {checkpoint_path}')
    for file_name in ('config_runtime.yaml', 'config_resolved.yaml'):
        candidate = seed_dir / file_name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'No config_runtime.yaml or config_resolved.yaml beside {checkpoint_path}')


def _sync_tree(src: Path, dst: Path, delete: bool = False) -> None:
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    rsync = shutil.which('rsync')
    if rsync is not None:
        cmd = [rsync, '-a']
        if delete:
            cmd.append('--delete')
        cmd += [f'{src}/', f'{dst}/']
        subprocess.run(cmd, check=True)
        return

    if delete and dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)

    for root, _, files in os.walk(src):
        rel = Path(root).relative_to(src)
        (dst / rel).mkdir(parents=True, exist_ok=True)
        for file_name in files:
            shutil.copy2(Path(root) / file_name, dst / rel / file_name)


def _write_checkpoint_aliases(
    run_dir: Path,
    metric: str = 'train_return_mean',
    maximize: bool = True,
) -> dict[str, str | None]:
    ckpt_dir = Path(run_dir) / 'checkpoints'
    ckpts = sorted(ckpt_dir.glob('epoch_*.pt')) if ckpt_dir.exists() else []
    aliases = {'last_checkpoint': None, 'best_checkpoint': None}
    if not ckpts:
        return aliases

    last_dst = ckpt_dir / 'last.pt'
    shutil.copy2(ckpts[-1], last_dst)
    aliases['last_checkpoint'] = str(last_dst)

    metrics_path = Path(run_dir) / 'metrics.csv'
    by_epoch = {p.stem.removeprefix('epoch_'): p for p in ckpts}
    best_ckpt = None
    best_row = None
    best_score = None

    if metrics_path.exists():
        with metrics_path.open('r', encoding='utf-8', newline='') as f:
            for row in csv.DictReader(f):
                raw_epoch = row.get('epoch') or row.get('iteration')
                raw_value = row.get(metric)
                if raw_epoch is None or raw_value in (None, '', 'nan', 'NaN'):
                    continue
                try:
                    epoch_key = f'{int(float(raw_epoch)):04d}'
                    score = float(raw_value)
                except ValueError:
                    continue
                ckpt = by_epoch.get(epoch_key)
                if ckpt is None:
                    continue
                if best_score is None or ((score > best_score) if maximize else (score < best_score)):
                    best_score = score
                    best_row = row
                    best_ckpt = ckpt

    if best_ckpt is not None:
        best_dst = ckpt_dir / 'best.pt'
        shutil.copy2(best_ckpt, best_dst)
        aliases['best_checkpoint'] = str(best_dst)
        with (ckpt_dir / 'best_checkpoint_info.json').open('w', encoding='utf-8') as f:
            json.dump(
                {
                    'source_checkpoint': str(best_ckpt),
                    'metric': metric,
                    'maximize': maximize,
                    'metric_row': best_row,
                },
                f,
                indent=2,
            )

    return aliases


def run_training(
    config_path: str | Path,
    seed: int | None = 0,
    output_tag: str | None = None,
    local_runs_root: Path | str = LOCAL_RUNS_ROOT,
    drive_runs_root: Path | str = DRIVE_RUNS_ROOT,
    sync_every_seconds: int = 300,
    periodic_sync: bool = True,
    best_metric: str = 'train_return_mean',
    maximize_metric: bool = True,
    extra_args: dict[str, Any] | None = None,
    resume_from: str | Path | None = None,
    overwrite: bool = True,
) -> dict[str, Any]:
    local_runs_root = Path(local_runs_root)
    drive_runs_root = Path(drive_runs_root)

    resume_checkpoint = None
    resolved_output_tag = output_tag
    if resume_from is not None:
        resume_checkpoint = _resolve_existing_path(resume_from, local_runs_root, drive_runs_root)
        if not resume_checkpoint.exists():
            raise FileNotFoundError(f'Resume checkpoint not found: {resume_checkpoint}')
        if seed is None:
            seed = _infer_seed_from_checkpoint(resume_checkpoint)
        if resolved_output_tag is None:
            resolved_output_tag = _default_resume_output_tag(resume_checkpoint)

    if seed is None:
        seed = 0

    stem = Path(config_path).stem
    if resume_checkpoint is not None:
        source_stem = _infer_run_stem_from_checkpoint(resume_checkpoint)
        if source_stem:
            stem = source_stem

    run_name = f"{stem}{'_' + resolved_output_tag if resolved_output_tag else ''}/seed_{seed}"
    local_run_dir = local_runs_root / run_name
    drive_run_dir = drive_runs_root / run_name
    local_run_dir.mkdir(parents=True, exist_ok=True)
    drive_run_dir.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        str(TRAIN_SCRIPT),
        '--config', str(config_path),
        '--seed', str(seed),
        '--output-dir', str(local_run_dir),
    ]
    if overwrite:
        cmd.append('--overwrite')
    if resume_checkpoint is not None:
        cmd.extend(['--resume-from', str(resume_checkpoint)])
    _add_cli_args(cmd, extra_args)

    print('Launching run:')
    _show_command(cmd)
    print('Local run dir :', local_run_dir)
    print('Drive run dir :', drive_run_dir)
    if resume_checkpoint is not None:
        print('Resume ckpt   :', resume_checkpoint)

    stop_event = threading.Event()

    def sync_worker() -> None:
        while not stop_event.wait(sync_every_seconds):
            try:
                _sync_tree(local_run_dir, drive_run_dir, delete=False)
                print(f'[sync] periodic sync complete -> {drive_run_dir}')
            except Exception as exc:
                print(f'[sync] periodic sync failed: {exc}')

    sync_thread = None
    if periodic_sync:
        sync_thread = threading.Thread(target=sync_worker, daemon=True)
        sync_thread.start()

    started_at = datetime.now(timezone.utc).isoformat()
    start_time = time.time()
    proc = subprocess.Popen(cmd, cwd=str(REPO_ROOT))
    return_code = None
    interrupted = False

    try:
        return_code = proc.wait()
    except KeyboardInterrupt:
        interrupted = True
        print('\n[run] Interrupted. Terminating training process before final sync...')
        proc.send_signal(signal.SIGINT)
        try:
            return_code = proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            return_code = proc.wait()
    finally:
        stop_event.set()
        if sync_thread is not None:
            sync_thread.join(timeout=5)

    ended_at = datetime.now(timezone.utc).isoformat()
    elapsed = time.time() - start_time
    if return_code != 0:
        print(f'[run] Training exited with code {return_code}. Performing final sync anyway...')

    aliases = _write_checkpoint_aliases(local_run_dir, metric=best_metric, maximize=maximize_metric)
    summary_path = local_run_dir / 'run_manager_summary.json'
    manager_status = 'interrupted' if interrupted else ('succeeded' if return_code == 0 else 'failed')
    summary = {
        'manager_status': manager_status,
        'return_code': return_code,
        'config_path': str(config_path),
        'seed': seed,
        'output_tag': resolved_output_tag,
        'run_name': run_name,
        'local_run_dir': str(local_run_dir),
        'drive_run_dir': str(drive_run_dir),
        'command': cmd,
        'command_pretty': (' ' + '\\' + '\n  ').join(cmd),
        'extra_args': extra_args or {},
        'resume_from': None if resume_checkpoint is None else str(resume_checkpoint),
        'resume_checkpoint_epoch': None if resume_checkpoint is None else _checkpoint_epoch_hint(resume_checkpoint),
        'overwrite': overwrite,
        'started_at_utc': started_at,
        'ended_at_utc': ended_at,
        'elapsed_sec': elapsed,
        'periodic_sync': periodic_sync,
        'sync_every_seconds': sync_every_seconds,
        'best_metric': best_metric,
        'maximize_metric': maximize_metric,
        'aliases': aliases,
    }
    with summary_path.open('w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    final_sync_error = None
    try:
        _sync_tree(local_run_dir, drive_run_dir, delete=False)
        print(f'[sync] final sync complete -> {drive_run_dir}')
    except Exception as exc:
        final_sync_error = repr(exc)
        print(f'[sync] final sync failed: {exc}')

    result = {
        'return_code': return_code,
        'manager_status': manager_status,
        'elapsed_sec': elapsed,
        'local_run_dir': str(local_run_dir),
        'drive_run_dir': str(drive_run_dir),
        'summary_path': str(summary_path),
        'resume_from': None if resume_checkpoint is None else str(resume_checkpoint),
        'final_sync_error': final_sync_error,
        **aliases,
    }
    print(json.dumps(result, indent=2))
    return result


def resume_training(
    checkpoint_path: str | Path,
    final_epochs: int,
    config_path: str | Path | None = None,
    seed: int | None = None,
    output_tag: str | None = None,
    preset_args: dict[str, Any] | None = None,
    extra_args: dict[str, Any] | None = None,
    **run_kwargs,
) -> dict[str, Any]:
    """Resume a run from a checkpoint and train until `final_epochs`.

    If `config_path` is omitted, the helper uses `config_runtime.yaml` from the
    checkpoint's run folder. Keep passing a preset such as `ATARI_TRPO_SAFE_ARGS`
    when you need CLI-only settings like `--device cuda`.
    """
    checkpoint_path = _resolve_existing_path(
        checkpoint_path,
        run_kwargs.get('local_runs_root', LOCAL_RUNS_ROOT),
        run_kwargs.get('drive_runs_root', DRIVE_RUNS_ROOT),
    )
    if config_path is None:
        config_path = config_path_for_checkpoint(
            checkpoint_path,
            run_kwargs.get('local_runs_root', LOCAL_RUNS_ROOT),
            run_kwargs.get('drive_runs_root', DRIVE_RUNS_ROOT),
        )
    resume_args = {}
    if preset_args:
        resume_args.update(preset_args)
    if extra_args:
        resume_args.update(extra_args)
    resume_args['--epochs'] = int(final_epochs)
    return run_training(
        config_path=config_path,
        seed=seed,
        output_tag=output_tag,
        extra_args=resume_args,
        resume_from=checkpoint_path,
        **run_kwargs,
    )


def run_batch(batch_spec, output_tag: str | None = None) -> list[dict[str, Any]]:
    results = []
    for config_path, seeds, preset_args, extra in batch_spec:
        for seed in seeds:
            print('\n' + '=' * 100)
            print(f'Running: config={config_path} seed={seed}')
            result = run_training(
                config_path=config_path,
                seed=seed,
                output_tag=output_tag,
                extra_args={**preset_args, **extra},
            )
            results.append(result)
    return results



def _relative_run_path(seed_dir: Path, runs_root: Path) -> Path:
    try:
        return seed_dir.relative_to(runs_root)
    except ValueError:
        return Path(seed_dir.parent.name) / seed_dir.name


def stitch_resumed_run(
    source_run_dir: str | Path,
    resume_run_dir: str | Path,
    resume_epoch: int | None = None,
    output_tag: str | None = 'stitched',
    output_dir: str | Path | None = None,
    local_runs_root: Path | str = LOCAL_RUNS_ROOT,
    drive_runs_root: Path | str = DRIVE_RUNS_ROOT,
    best_metric: str = 'train_return_mean',
    maximize_metric: bool = True,
    copy_checkpoints: bool = True,
    overwrite: bool = True,
    sync_to_drive: bool = True,
) -> dict[str, Any]:
    """Create a compiled seed folder from an original run plus a resumed run.

    The stitched metrics keep source rows up to `resume_epoch` and resumed rows
    after `resume_epoch`, which discards duplicate partial rows from the failed
    source run. The output folder is usable with `aggregate_runs(...)`.
    """
    local_runs_root = Path(local_runs_root)
    drive_runs_root = Path(drive_runs_root)
    source_dir = _resolve_existing_path(source_run_dir, local_runs_root, drive_runs_root)
    resume_dir = _resolve_existing_path(resume_run_dir, local_runs_root, drive_runs_root)
    if not (source_dir / 'metrics.csv').exists():
        raise FileNotFoundError(f'Source metrics.csv not found under: {source_dir}')
    if not (resume_dir / 'metrics.csv').exists():
        raise FileNotFoundError(f'Resume metrics.csv not found under: {resume_dir}')

    if output_dir is None:
        source_stem = source_dir.parent.name
        seed_name = source_dir.name
        stitched_stem = source_stem if output_tag in (None, '') else f'{source_stem}_{output_tag}'
        local_output_dir = local_runs_root / stitched_stem / seed_name
    else:
        local_output_dir = Path(output_dir)

    relative_output = _relative_run_path(local_output_dir, local_runs_root)
    drive_output_dir = drive_runs_root / relative_output

    cmd = [
        sys.executable,
        str(STITCH_SCRIPT),
        '--source-run-dir', str(source_dir),
        '--resume-run-dir', str(resume_dir),
        '--output-dir', str(local_output_dir),
        '--metric', best_metric,
    ]
    if resume_epoch is not None:
        cmd.extend(['--resume-epoch', str(resume_epoch)])
    if overwrite:
        cmd.append('--overwrite')
    if not maximize_metric:
        cmd.append('--minimize')
    if not copy_checkpoints:
        cmd.append('--no-copy-checkpoints')

    print('Stitching resumed run:')
    _show_command(cmd)
    print('Source run dir:', source_dir)
    print('Resume run dir:', resume_dir)
    print('Local output  :', local_output_dir)
    print('Drive output  :', drive_output_dir)
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)

    final_sync_error = None
    if sync_to_drive:
        try:
            _sync_tree(local_output_dir, drive_output_dir, delete=False)
            print(f'[sync] stitched run synced -> {drive_output_dir}')
        except Exception as exc:
            final_sync_error = repr(exc)
            print(f'[sync] stitched sync failed: {exc}')

    stitch_metadata_path = local_output_dir / 'stitch_metadata.json'
    stitch_metadata = {}
    if stitch_metadata_path.exists():
        with stitch_metadata_path.open('r', encoding='utf-8') as f:
            stitch_metadata = json.load(f)

    result = {
        'local_run_dir': str(local_output_dir),
        'drive_run_dir': str(drive_output_dir),
        'metrics_csv': str(local_output_dir / 'metrics.csv'),
        'stitch_metadata_path': str(stitch_metadata_path),
        'final_sync_error': final_sync_error,
        **stitch_metadata,
    }
    print(json.dumps(result, indent=2))
    return result

def aggregate_runs(
    runs_roots: Iterable[str | Path],
    compare: bool = False,
    metric: str = 'train_return_mean',
    x_axis: str = 'iteration',
    smooth_window: int | None = None,
    summary: bool = True,
):
    cmd = [sys.executable, str(AGGREGATE_SCRIPT)]
    for root in runs_roots:
        cmd.extend(['--runs-root', str(root)])
    if compare:
        cmd.append('--compare')
    cmd.extend(['--metric', metric, '--x-axis', x_axis])
    if smooth_window is not None:
        cmd.extend(['--smooth-window', str(smooth_window)])
    if summary:
        cmd.append('--summary')
    _show_command(cmd)
    return subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)


def evaluate_checkpoint(
    config_path: str,
    checkpoint_path: str | Path,
    episodes: int = 5,
    device: str = 'cuda',
    extra_args: dict[str, Any] | None = None,
):
    cmd = [
        sys.executable,
        str(EVAL_SCRIPT),
        '--config', config_path,
        '--checkpoint', str(checkpoint_path),
        '--episodes', str(episodes),
        '--device', device,
    ]
    _add_cli_args(cmd, extra_args)
    _show_command(cmd)
    return subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)


## 7. Recommended argument presets

Run this cell once after the helper cell. These dictionaries are passed into `run_training(extra_args=...)`, so this is where you usually adjust device, workers, memory mode, and other common command-line options.


In [ ]:

ATARI_TRPO_SAFE_ARGS = {
    '--num-workers': 4,
    '--memory-mode': 'safe',
    '--obs-storage': 'ram',
    '--full-batch-chunk-size': 8192,
    '--device': 'cuda',
    '--progress-mode': 'off',
}

ATARI_PPO_ARGS = {
    '--num-workers': 4,
    '--memory-mode': 'standard',
    '--device': 'cuda',
    '--progress-mode': 'off',
}

MUJOCO_TRPO_ARGS = {
    '--num-workers': 12,
    '--memory-mode': 'standard',
    '--device': 'cuda',
    '--progress-mode': 'off',
}

MUJOCO_NPG_ARGS = {
    '--num-workers': 12,
    '--memory-mode': 'standard',
    '--device': 'cuda',
    '--progress-mode': 'off',
}

MUJOCO_PPO_ARGS = {
    '--num-workers': 12,
    '--memory-mode': 'standard',
    '--device': 'cuda',
    '--progress-mode': 'off',
}


## 8. Single-run launcher

This is the main place to start one training run or resume a partial one. Pick a recipe or edit one, then uncomment either `run_training(...)` for a fresh run or `resume_training(...)` for a checkpoint resume in the next cell.


In [ ]:
# Main single-run launcher.
# Edit SELECTED_RUN, or edit one of the recipes below, then uncomment the launch line you need.
SINGLE_RUN_RECIPES = {
    'seaquest_trpo_single_path': {
        'config_path': 'configs/atari/seaquest_single_path.yaml',
        'seed': 0,
        'output_tag': None,
        'extra_args': {**ATARI_TRPO_SAFE_ARGS, '--epochs': 300, '--save-interval': 25},
    },
    'seaquest_ppo_clip': {
        'config_path': 'configs/atari/seaquest_ppo_clip.yaml',
        'seed': 0,
        'output_tag': None,
        'extra_args': {**ATARI_PPO_ARGS, '--epochs': 300, '--save-interval': 25},
    },
    'walker2d_trpo_single_path': {
        'config_path': 'configs/mujoco/walker2d_single_path.yaml',
        'seed': 0,
        'output_tag': None,
        'extra_args': {**MUJOCO_TRPO_ARGS, '--epochs': 200, '--save-interval': 25},
    },
}

RESUME_RECIPES = {
    'pong_trpo_epoch_250_to_300': {
        # Relative paths are resolved under LOCAL_RUNS_ROOT first, then DRIVE_RUNS_ROOT.
        'checkpoint_path': 'pong_single_path/seed_0/checkpoints/epoch_0250.pt',
        # If config_path is None, resume_training uses config_runtime.yaml beside the checkpoint.
        'config_path': None,
        'seed': None,  # infer from the checkpoint folder name, e.g. seed_0
        'final_epochs': 300,
        'output_tag': None,  # defaults to resume_0250
        'preset_args': ATARI_TRPO_SAFE_ARGS,
        'extra_args': {'--save-interval': 25},
    },
}

SELECTED_RUN = 'seaquest_ppo_clip'
recipe = SINGLE_RUN_RECIPES[SELECTED_RUN]

print('Selected run:', SELECTED_RUN)
print('Config      :', recipe['config_path'])
print('Seed        :', recipe['seed'])
print('Uncomment the fresh-run line when you are ready to launch.')

# result = run_training(**recipe)
# result

# To resume Pong from epoch 250 to epoch 300, use this instead:
# resume_recipe = RESUME_RECIPES['pong_trpo_epoch_250_to_300']
# resume_result = resume_training(**resume_recipe)
# resume_result


## 9. Batch launcher

Use this when you want to run several configs or seeds. Edit the batch plans first, then uncomment one launch line at the bottom of the next cell.


In [ ]:
# Batch launch cell. Each tuple is:
# (config_path, seeds, preset_args, run_specific_extra_args)
LOCOMOTION_BATCH = [
    ('configs/mujoco/swimmer_single_path.yaml', [0, 1, 2], MUJOCO_TRPO_ARGS, {'--epochs': 200, '--save-interval': 25}),
    ('configs/mujoco/hopper_single_path.yaml', [0, 1, 2], MUJOCO_TRPO_ARGS, {'--epochs': 200, '--save-interval': 25}),
    ('configs/mujoco/walker2d_single_path.yaml', [0, 1, 2], MUJOCO_TRPO_ARGS, {'--epochs': 200, '--save-interval': 25}),
]

ATARI_BATCH = [
    ('configs/atari/beamrider_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/breakout_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/enduro_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/pong_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/qbert_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/seaquest_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
    ('configs/atari/spaceinvaders_single_path.yaml', [0], ATARI_TRPO_SAFE_ARGS, {'--epochs': 300, '--save-interval': 25}),
]

print('Locomotion runs:', sum(len(seeds) for _, seeds, _, _ in LOCOMOTION_BATCH))
print('Atari runs     :', sum(len(seeds) for _, seeds, _, _ in ATARI_BATCH))
print('Uncomment one launch line when you are ready.')

# locomotion_results = run_batch(LOCOMOTION_BATCH)
# atari_results = run_batch(ATARI_BATCH)


## 10. Post-run stitching, aggregation, and evaluation

Run these after training has produced `metrics.csv` and checkpoints. Stitching can combine an original failed run with a resumed run into one clean `metrics.csv`; aggregation reads run folders; evaluation loads a checkpoint such as `checkpoints/best.pt` or `checkpoints/last.pt`.


In [ ]:
# Example stitching for Pong TRPO: keep old epochs 1-250 and resumed epochs 251-300.
# The output can be graphed with aggregate_runs just like a normal run.
# stitched = stitch_resumed_run(
#     source_run_dir='pong_single_path/seed_0',
#     resume_run_dir='pong_single_path_resume_0250/seed_0',
#     resume_epoch=250,
#     output_tag='stitched',
#     best_metric='train_return_mean',
#     maximize_metric=True,
# )
#
# aggregate_runs(
#     runs_roots=[Path(stitched['local_run_dir']).parent],
#     compare=False,
#     metric='train_return_mean',
#     x_axis='epoch',
#     smooth_window=5,
#     summary=True,
# )

# Example aggregation for a locomotion comparison.
# Change these folders to the runs you want to compare.
# aggregate_runs(
#     runs_roots=[
#         LOCAL_RUNS_ROOT / 'swimmer_single_path',
#         LOCAL_RUNS_ROOT / 'swimmer_natural_pg',
#         LOCAL_RUNS_ROOT / 'swimmer_ppo_clip',
#     ],
#     compare=True,
#     metric='train_return_mean',
#     x_axis='iteration',
#     smooth_window=5,
#     summary=True,
# )


In [ ]:
# Example evaluation of a finished run's best checkpoint.
# Change run_dir/config_path if you trained a different environment or policy.
# run_dir = LOCAL_RUNS_ROOT / 'seaquest_single_path' / 'seed_0'
# best_ckpt = run_dir / 'checkpoints' / 'best.pt'
# evaluate_checkpoint(
#     config_path='configs/atari/seaquest_single_path.yaml',
#     checkpoint_path=best_ckpt,
#     episodes=5,
#     device='cuda',
# )


## 11. Notes / practical advice

- The notebook is standalone: all Run Manager code lives in the helper cell above.
- The helper and preset cells should be run once per Colab runtime. Rerun them only if you edit them.
- The actual training launch points are section 8 for one run/resume and section 9 for batches.
- For checkpoint resume, `final_epochs` is the final target epoch. Resuming epoch 250 with `final_epochs=300` runs epochs 251 through 300.
- Resume outputs default to a new run folder like `pong_single_path_resume_0250/seed_0`, so the partial source run is not overwritten.
- Stitched outputs default to a graphable folder like `pong_single_path_stitched/seed_0`; use that folder for final aggregation/reporting.
- Periodic sync reduces the chance of losing checkpoints if the runtime disconnects.
- Final sync runs after the training subprocess exits, even when the return code is non-zero.
- `best.pt` is chosen only from epochs that have a saved checkpoint file; `last.pt` is the latest saved checkpoint file. Stitching copies matching epoch checkpoints when available, then regenerates these aliases.
- For Atari TRPO on Colab, `memory_mode=safe` + `obs_storage=ram` + chunking is still the intended setup.


## 12. More Notes

- For **Atari**, start with `memory_mode safe`.
- If Colab has plenty of RAM, `obs_storage ram` is usually faster than memmap.
- Tune `full_batch_chunk_size` conservatively: `4096`, `8192`, then maybe `16384`.
- If using TRPO/NPG, `fvp_subsample_fraction 0.1` is a useful optional speedup.
- Run from `/content/trpo`, not directly from mounted Drive, for better performance.